In [36]:
from selenium import webdriver
from selenium.common.exceptions import WebDriverException
import os
from tqdm import tqdm
import pandas as pd
import time

In [37]:
def get_redirect_link(url):
    """
    Resolves a redirect URL to its final destination URL using Selenium.

    Args:
      url: The initial URL that may be a redirect.

    Returns:
      The final destination URL after following redirects, or the original URL
      if an error occurred.
    """
    # Set up the WebDriver (e.g., ChromeDriver). Make sure the driver executable
    # is in your system's PATH or provide the path to the executable.
    # You might need to change this based on the browser you have installed (e.g., Firefox, Edge).
    driver = None  # Initialize driver to None
    try:
        # Using Chrome as an example. You might need options like --headless
        options = webdriver.ChromeOptions()
        # if you don't want a browser window to open.
        # Uncomment the line below to run in headless mode (no browser window)
        # options.add_argument("--headless")
        # options.add_argument('--no-sandbox') # Recommended for some environments
        # options.add_argument('--disable-dev-shm-usage') # Recommended for some environments

        driver = webdriver.Chrome(options=options)
        # print(f"Attempting to resolve URL: {url}")
        driver.get(url)

        # Wait for the body element to be present. This is a more reliable indicator
        # that the page has loaded after redirects compared to checking the title.
        # print("sleeping for 1 seconds to allow for any additional redirects...")
        time.sleep(2)
        # Stop the driver from loading further by executing a script to stop network activity
        # print("Stopping the driver from loading further...")
        driver.execute_script("window.stop();")
        # Get the current URL after all redirects have occurred
        final_url = driver.current_url
        # print(f"Final URL resolved by Selenium: {final_url}")

        return final_url

    except WebDriverException:
        # print(f"An error occurred with Selenium: {e}")
        return url  # Return original URL if an error occurs
    finally:
        # Always close the browser session if the driver was successfully initialized
        if driver:
            driver.quit()
            # print("Browser session closed.")

In [38]:
news_data_directory = "../data/news/"
output_directory = "../data/news_with_proper_links/"

news_csv_files = [x for x in os.listdir(news_data_directory) if x.endswith(".csv")]

In [39]:
df

,title,date,link,final_url
0,Click here to print,2015-01-02T08:00:00Z,https://news.google.com/read/CBMicEFVX3lxTE1yN...,https://www.google.com/sorry/index?continue=ht...
1,Bitter taste of sugar,2015-01-03T08:00:00Z,https://news.google.com/read/CBMiSEFVX3lxTE5Ta...,None
2,EDITORIAL: Manasa Vaniqi Leaves A Legacy Of Se...,2015-01-03T08:00:00Z,https://news.google.com/read/CBMiogFBVV95cUxPO...,None
3,Batéy 106: Portraits from a Dominican Sugar Ca...,2015-01-04T08:00:00Z,https://news.google.com/read/CBMidEFVX3lxTE5lY...,None
4,Expectations of lower sugar imports amid start...,2015-01-04T08:00:00Z,https://news.google.com/read/CBMiqAFBVV95cUxOb...,None
...,...,...,...,...
16677,New Study Finds Preharvest Sugarcane Burns Are...,2025-05-01T04:20:55Z,https://news.google.com/read/CBMixAFBVV95cUxNM...,None
16678,Sugarcane farmers and workers’ unions pledge t...,2025-05-01T05:25:36Z,https://news.google.com/read/CBMivwFBVV95cUxQZ...,None
16679,Union government hikes fair and remunerative p...,2025-05-01T05:42:00Z,https://news.google.com/read/CBMirgFBVV95cUxQb...,None
16680,Cabinet Hikes Sugarcane Price for 2025–26 Season,2025-05-01T05:50:15Z,https://news.google.com/read/CBMijAFBVV95cUxPL...,None


In [35]:
for file in news_csv_files:
    commodity_name = file.split("_")[1]
    print(f"Processing file: {file} for commodity: {commodity_name}")
    df = pd.read_csv(os.path.join(news_data_directory, file))
    df["final_url"] = None
    pbar = tqdm(total=len(df), desc=f"Processing {commodity_name}", unit="row")
    for index, row in df.iterrows():
        print(f"Processing row {index} for commodity: {commodity_name}")
        df.at[index, "final_url"] = get_redirect_link(row["link"])
        print(f"Row {index} final URL: {df.at[index, 'final_url']}")
        pbar.update(1)
    df.to_csv(os.path.join(output_directory, file), index=False)

Processing file: news_sugarcane_2015-01-01_to_2025-04-30.csv for commodity: sugarcane


Processing sugarcane:   0%|          | 3/16682 [01:54<177:06:08, 38.23s/row]

Processing row 0 for commodity: sugarcane



Processing sugarcane:   0%|          | 1/16682 [00:05<24:18:45,  5.25s/row]

Row 0 final URL: https://www.google.com/sorry/index?continue=https://news.google.com/read/CBMicEFVX3lxTE1yNFNiTW4tX19pQTVkcUpZdTdwRV9ERll6cXFCYUk0TVNHQ2xINUVvVWpzbkVuZFhmbkF6bnFqYkplZWVFTkxiMTJVQzlPUUVDSHl5YUdNTXhBN2oyZkxmaVltdVJ4MDl2WDlKa1JQQzI%3Fhl%3Den-US%26gl%3DUS%26ceid%3DUS%253Aen&hl=en-US&q=EgSlhGh2GJ6p9cAGIjCKvdM6XzuFbJz-FdGdkyIu9DsNLMNCUAOteCzT_phcI3_LfCstOAyZ1zjDGqjDjusyAnJSWgFD
Processing row 1 for commodity: sugarcane


KeyboardInterrupt: 